![](https://www.kaggle.com/competitions/129329/images/header)

<div style="
    padding: 10px;
    background-color: #FFFFFF;
    color:blue;
    border-radius: 3px;
">
<h2>Table of Contents </h2>
    <ol>
        <li>Dataset Overview</li>
        <li>Taxonomy</li>
        <li>Class imbalance</li>
        <li>Recording locations</li>
        <li>Audio quality</li>
        <li>Secondary labels</li>
        <li>Audio visualization</li>
        <li>Spectrogram comparison</li>
        <li>Soundscape labels analysis</li>
        <li>Audio length</li>
        <li>Frequency profile comparison</li>
    </ol>
</div>


In [ ]:
import numpy as np
import pandas as pd
import os
import random

import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.io as pio

import librosa
import librosa.display

from IPython.display import Audio

sns.set_style("whitegrid")

#-------------------------------------------

DATA_PATH = "/kaggle/input/competitions/birdclef-2026/"
AUDIO_PATH = DATA_PATH + "train_audio/"

<div style="
    padding: 10px;
    background-color: #FFFFFF;
    color:blue;
    border-radius: 3px;
">
<h2>Dataset Overview </h2>
</div>

In [ ]:
train = pd.read_csv(DATA_PATH + "train.csv")
taxonomy = pd.read_csv(DATA_PATH + "taxonomy.csv")
labels = pd.read_csv(DATA_PATH + "train_soundscapes_labels.csv")

sample_submission = pd.read_csv(DATA_PATH + "sample_submission.csv")

recording_location = DATA_PATH + "recording_location.txt"

print("train.csv:", train.shape)
print("taxonomy.csv:", taxonomy.shape)
print("train_soundscapes_labels.csv:", labels.shape)
print("sample_submission.csv:", sample_submission.shape)
train.head()

In [ ]:
print("Num species:", train.primary_label.nunique())
print("Collections:")
print(train.collection.value_counts())

In [ ]:
summary = pd.DataFrame({
    "metric": [
        "Number of training clips",
        "Number of unique species in train.csv",
        "Number of classes in taxonomy",
        "Number of soundscape annotations",
        "Number of submission target columns"
    ],
    "value": [
        len(train),
        train["primary_label"].nunique(),
        len(taxonomy),
        len(labels),
        sample_submission.shape[1] - 1
    ]
})

summary

## 🔎 First impression

A quick sanity check already tells us something important:

- `taxonomy.csv` defines the final target space
- `train.csv` covers clip-level metadata
- `train_soundscapes_labels.csv` is crucial because it is closest to test-time conditions
- submission is **multi-label probability prediction** across **234 species columns**

This means a good competition solution probably cannot rely on `train_audio` alone.

<div style="
    padding: 10px;
    background-color: #FFFFFF;
    color:blue;
    border-radius: 3px;
">
<h2>Taxonomy & Class imbalance</h2>
</div>

In [ ]:
taxonomy.head()

In [ ]:
taxonomy["class_name"].value_counts()

In [ ]:
plt.figure(figsize=(6,4))
sns.countplot(x="class_name", data=taxonomy)
plt.title("Class distribution")
plt.show()

## Insight

This is not a bird-only challenge.

We are modeling a mixed acoustic ecology problem:
- **Aves**
- **Insecta**
- **Mammalia**
- **Reptilia**
- **Amphibia**

That makes the frequency landscape more diverse and likely increases confusion.


In [ ]:
#Merge taxonomy into train
train = train.merge(
    taxonomy[["primary_label", "class_name"]],
    on="primary_label",
    how="left"
)

train.head(2)

In [ ]:
species_counts = train["primary_label"].value_counts()
pio.renderers.default = 'iframe'
fig = px.histogram(
    x=species_counts.values,
    nbins=50,
    title="Distribution of number of clips per species"
)
fig.update_layout(xaxis_title="Number of training clips for a species",
                  yaxis_title="Count of species")
fig.show()

In [ ]:
top_species = species_counts.head(25).sort_values()

plt.figure(figsize=(10, 8))
plt.barh(top_species.index, top_species.values)
plt.title("Top 25 most frequent species in training set")
plt.xlabel("Number of clips")
plt.ylabel("Species")
plt.show()

In [ ]:
bottom_species = species_counts.tail(25).sort_values()

plt.figure(figsize=(10, 8))
plt.barh(bottom_species.index, bottom_species.values)
plt.title("25 rarest species in training set")
plt.xlabel("Number of clips")
plt.ylabel("Species")
plt.show()

## Insight

The training set is heavily imbalanced.

That matters because:
- common species may dominate optimization
- rare species are easy to underfit
- calibration may be poor on tail classes
- sampling strategy may matter as much as model architecture

Typical responses:
- weighted BCE / focal loss
- class-aware sampling
- oversampling rare classes
- pseudo-labeling from soundscapes

<div style="
    padding: 10px;
    background-color: #FFFFFF;
    color:blue;
    border-radius: 3px;
">
<h2>Recording locations</h2>
</div>

In [ ]:
train["collection"].value_counts()

In [ ]:
pio.renderers.default = 'iframe'
fig = px.pie(
    names=train["collection"].value_counts().index,
    values=train["collection"].value_counts().values,
    title="Collection sources in train_audio"
)
fig.show()

In [ ]:
pio.renderers.default = 'iframe'


fig = px.scatter_geo(
    train,
    lat="latitude",
    lon="longitude",
    color="collection",
    title="Recording locations"
)

fig.show()

In [ ]:
geo = train.dropna(subset=["latitude", "longitude"]).copy()
pio.renderers.default = 'iframe' # lub 'notebook_connected'
fig = px.scatter_geo(
    geo,
    lat="latitude",
    lon="longitude",
    color="class_name_x",
    hover_name="primary_label",
    title="Geographic distribution of recordings",
    opacity=0.7
)
fig.show()

In [ ]:
fig = px.scatter_map(
    geo.sample(min(5000, len(geo))),
    lat="latitude",
    lon="longitude",
    color="class_name_x",
    hover_name="primary_label",
    zoom=4,
    map_style="open-street-map",
    title="Sampled recording map"
)
fig.show()

## Insight

Even within one region, geography may still matter.

Possible reasons:
- habitat differences
- local acoustic communities
- temporal activity patterns by site
- call dialects in some bird species

<div style="
    padding: 10px;
    background-color: #FFFFFF;
    color:blue;
    border-radius: 3px;
">
<h2>Audio quality</h2>
</div>

In [ ]:
train["rating"] = pd.to_numeric(train["rating"], errors="coerce")

plt.figure(figsize=(10, 5))
sns.histplot(train["rating"].dropna(), bins=20, kde=True)
plt.title("Distribution of user-provided quality ratings")
plt.xlabel("Rating")
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))
sns.boxplot(data=train, x="class_name_x", y="rating")
plt.title("Rating distribution by class")
plt.show()

## Insight

`rating` can be valuable:
- high-rating clips may be useful for cleaner warm-up training
- lower-rating clips may improve robustness

<div style="
    padding: 10px;
    background-color: #FFFFFF;
    color:blue;
    border-radius: 3px;
">
<h2>Secondary labels</h2>
</div>

In [ ]:
def parse_secondary_labels(x):
    if pd.isna(x):
        return []
    if isinstance(x, list):
        return x
    try:
        return ast.literal_eval(x)
    except:
        return []

train["secondary_list"] = train["secondary_labels"].apply(parse_secondary_labels)
train["n_secondary"] = train["secondary_list"].apply(len)

train[["primary_label", "secondary_labels", "n_secondary"]].head()

## Insight
No Secondary labels

<div style="
    padding: 10px;
    background-color: #FFFFFF;
    color:blue;
    border-radius: 3px;
">
<h2>Audio visualization</h2>
</div>

In [ ]:
file = train.filename.iloc[0]

path = AUDIO_PATH + file

Audio(path)

In [ ]:
y, sr = librosa.load(path, sr=None)

plt.figure(figsize=(12,4))
librosa.display.waveshow(y, sr=sr)
plt.show()

<div style="
    padding: 10px;
    background-color: #FFFFFF;
    color:blue;
    border-radius: 3px;
">
<h2>Spectrogram comparison</h2>
</div>

In [ ]:
S = librosa.feature.melspectrogram(y=y, sr=sr)
S_db = librosa.power_to_db(S)

plt.figure(figsize=(10,4))
librosa.display.specshow(
    S_db,
    sr=sr,
    x_axis="time",
    y_axis="mel"
)
plt.colorbar()
plt.show()

In [ ]:
samples = train.primary_label.unique()[:4]

for sp in samples:

    f = train[train.primary_label == sp].filename.iloc[0]

    y, sr = librosa.load(AUDIO_PATH + f)

    S = librosa.feature.melspectrogram(y=y, sr=sr)
    S_db = librosa.power_to_db(S)

    plt.figure(figsize=(8,3))
    librosa.display.specshow(S_db, sr=sr)
    plt.title(sp)
    plt.show()

In [ ]:
def plot_waveform_and_mel(filepath, title="", sr=32000, n_mels=128, fmax=16000):
    y, sr = librosa.load(filepath, sr=sr)
    
    fig, axes = plt.subplots(2, 1, figsize=(14, 7))
    
    librosa.display.waveshow(y, sr=sr, ax=axes[0])
    axes[0].set_title(f"Waveform | {title}")
    
    mel = librosa.feature.melspectrogram(
        y=y, sr=sr, n_mels=n_mels, fmax=fmax
    )
    mel_db = librosa.power_to_db(mel, ref=np.max)
    
    img = librosa.display.specshow(
        mel_db, sr=sr, x_axis="time", y_axis="mel",
        fmax=fmax, ax=axes[1]
    )
    axes[1].set_title(f"Mel Spectrogram | {title}")
    fig.colorbar(img, ax=axes[1], format="%+2.0f dB")
    plt.tight_layout()
    plt.show()

In [ ]:
example_row = train.sample(1, random_state=1).iloc[0]
plot_waveform_and_mel(
    AUDIO_PATH + example_row["filename"],
    title=f"{example_row['primary_label']} ({example_row['class_name_x']})"
)

<div style="
    padding: 10px;
    background-color: #FFFFFF;
    color:blue;
    border-radius: 3px;
">
<h2>Soundscape labels analysis</h2>
</div>

In [ ]:
labels.head()

labels["num_labels"] = labels.primary_label.str.split(";").apply(len)

plt.hist(labels.num_labels)
plt.title("Labels per segment")
plt.show()

In [ ]:
labels["end"] = pd.to_timedelta(labels["end"])
labels["start"] = pd.to_timedelta(labels["start"])

labels["label_list"] = labels["primary_label"].fillna("").apply(
    lambda x: [i for i in x.split(";") if i != ""]
)
labels["n_labels"] = labels["label_list"].apply(len)
labels["duration"] = labels["end"] - labels["start"]

labels[["filename", "start", "end", "n_labels"]].head()

In [ ]:
labels["duration_seconds"] = labels["duration"].dt.total_seconds()
labels.duration_seconds.value_counts()

In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(labels["n_labels"], bins=20)
plt.title("Number of labels per soundscape segment")
plt.xlabel("Labels per 5-second segment")
plt.show()

In [ ]:
pio.renderers.default = 'iframe'
fig = px.histogram(
    labels,
    x="start",
    nbins=12,
    title="Distribution of segment start times in train_soundscapes_labels"
)
fig.show()

In [ ]:
files = random.sample(list(train.filename), 50)

lengths = []

for f in files:
    y, sr = librosa.load(AUDIO_PATH + f)
    lengths.append(len(y)/sr)

plt.hist(lengths)
plt.title("Audio length")
plt.show()

## Insight

Soundscape labels are valuable because they resemble the hidden test setting.

### Which species appear in soundscapes?

In [ ]:
all_ss_labels = labels["label_list"].explode()
all_ss_labels = all_ss_labels[all_ss_labels.notna() & (all_ss_labels != "")]

ss_species_counts = all_ss_labels.value_counts()

plt.figure(figsize=(10, 8))
top_ss = ss_species_counts.head(25).sort_values()
plt.barh(top_ss.index, top_ss.values)
plt.title("Top 25 most frequent labels in train_soundscapes")
plt.xlabel("Number of labeled 5-second segments")
plt.show()

In [ ]:


labels["num_labels"] = labels.primary_label.str.split(";").apply(len)

plt.hist(labels.num_labels)
plt.title("Labels per segment")
plt.show()

In [ ]:
species_in_train_audio = set(train["primary_label"].unique())
species_in_soundscape = set(all_ss_labels.unique())

print("Species in train_audio:", len(species_in_train_audio))
print("Species in labeled soundscapes:", len(species_in_soundscape))
print("Only in soundscapes:", len(species_in_soundscape - species_in_train_audio))
print("Only in train_audio:", len(species_in_train_audio - species_in_soundscape))

## Insight

Competition description already hints at something crucial:

Some test species may be represented **only in labeled soundscapes**, not in `train_audio`.

Ignoring soundscape labels may directly hurt performance?

<div style="
    padding: 10px;
    background-color: #FFFFFF;
    color:blue;
    border-radius: 3px;
">
<h2>Frequency profile comparison</h2>
</div>

In [ ]:
def mean_frequency_profile(filepath, sr=32000, n_mels=128, fmax=16000):
    y, sr = librosa.load(filepath, sr=sr)
    mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=n_mels, fmax=fmax)
    mel_db = librosa.power_to_db(mel, ref=np.max)
    return mel_db.mean(axis=1)

species_compare = train["primary_label"].drop_duplicates().sample(5, random_state=21).tolist()

plt.figure(figsize=(12, 6))
for species in species_compare:
    row = train[train["primary_label"] == species].sample(1, random_state=42).iloc[0]
    profile = mean_frequency_profile(AUDIO_PATH + row["filename"])
    plt.plot(profile, label=species)

plt.title("Average mel-bin energy profile for selected species")
plt.xlabel("Mel bin")
plt.ylabel("Mean dB")
plt.legend()
plt.show()